In [1]:
# DiffDock Batch Docking Pipeline - Simplified Version
#
# This notebook docks all protein-ligand combinations using DiffDock.
# Uses the standard DiffDock inference with CSV input for batch processing.
#
# Reference: https://github.com/gcorso/DiffDock
#
# Usage:
#   python -m inference --config default_inference_args.yaml \
#     --protein_ligand_csv input.csv --out_dir ./results
#
# For single complex:
#   python -m inference --protein_path protein.pdb --ligand ligand.sdf --out_dir ./output

In [2]:
from __future__ import annotations

import csv
import json
import os
import shutil
import subprocess
import time
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import numpy as np
from collections import defaultdict

In [3]:
# ============================================================================
# CONFIGURATION - Adjust these parameters as needed
# ============================================================================

# Paths
workspace_root = Path.cwd()
drugs_dir = workspace_root / "Drugs"
receptors_dir = workspace_root / "Orai"

# DiffDock paths
DIFFDOCK_DIR = Path("/home/manndo/docking_tools/DiffDock")
DIFFDOCK_CONDA_ENV = "diffdock"

# Number of samples (poses) DiffDock generates per complex
NUM_SAMPLES: int = 30

# Device for DiffDock (cpu or cuda:0)
DIFFDOCK_DEVICE = "cpu"

# Output directories
DIFFDOCK_OUTPUT_DIR = workspace_root / "diffdock_results"
DIFFDOCK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Overwrite settings
OVERWRITE_EXISTING = False  # Set to True to re-dock existing combinations

# Validate directories
if not drugs_dir.exists():
    raise FileNotFoundError(f"Ligand directory missing: {drugs_dir}")
if not receptors_dir.exists():
    raise FileNotFoundError(f"Receptor directory missing: {receptors_dir}")

print("=" * 80)
print("DiffDock Batch Docking Configuration")
print("=" * 80)
print(f"Number of samples per complex: {NUM_SAMPLES}")
print(f"DiffDock directory: {DIFFDOCK_DIR}")
print(f"DiffDock conda env: {DIFFDOCK_CONDA_ENV}")
print(f"DiffDock device: {DIFFDOCK_DEVICE}")
print(f"Output directory: {DIFFDOCK_OUTPUT_DIR}")
print(f"Overwrite existing: {OVERWRITE_EXISTING}")
print()

# Validate DiffDock installation
if not DIFFDOCK_DIR.exists():
    print(f"⚠️  WARNING: DiffDock directory not found at {DIFFDOCK_DIR}")
    print("   Please update DIFFDOCK_DIR to point to your DiffDock installation")
else:
    print(f"✓ DiffDock directory found")

DiffDock Batch Docking Configuration
Number of samples per complex: 30
DiffDock directory: /home/manndo/docking_tools/DiffDock
DiffDock conda env: diffdock
DiffDock device: cpu
Output directory: /home/manndo/MasterProject/diffdock_results
Overwrite existing: False

✓ DiffDock directory found


In [4]:
# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

@dataclass
class DockingResult:
    """Result of docking a protein-ligand combination."""
    protein_name: str
    ligand_name: str
    protein_path: Path
    ligand_path: Path
    output_dir: Path
    status: str  # "success", "failed", "skipped"
    num_poses: int = 0
    pose_files: List[Path] = field(default_factory=list)
    error_message: str = ""
    elapsed_time: float = 0.0
    
    def to_dict(self) -> dict:
        return {
            "protein_name": self.protein_name,
            "ligand_name": self.ligand_name,
            "protein_path": str(self.protein_path),
            "ligand_path": str(self.ligand_path),
            "output_dir": str(self.output_dir),
            "status": self.status,
            "num_poses": self.num_poses,
            "pose_files": [str(p) for p in self.pose_files],
            "error_message": self.error_message,
            "elapsed_time": self.elapsed_time,
        }


def get_file_stem(path: Path) -> str:
    """Get clean filename stem without extension."""
    return path.stem.replace("_ligand", "").replace("_protein", "")


def collect_files(root: Path, extensions: List[str]) -> List[Path]:
    """Return files with given extensions located directly inside root (no recursion)."""
    files = []
    for ext in extensions:
        files.extend(sorted(p for p in root.glob(f"*{ext}") if p.is_file()))
    return sorted(set(files))


def extract_confidence_from_filename(filename: str) -> float:
    """Extract confidence score from DiffDock output filename (e.g., rank1_confidence-0.85.sdf)."""
    try:
        if "confidence" in filename.lower():
            parts = filename.lower().split("confidence")
            if len(parts) > 1:
                conf_str = parts[1].replace("-", "").replace("_", "").replace(".sdf", "")
                return float(conf_str)
    except:
        pass
    return 0.0


def extract_rank_from_filename(filename: str) -> int:
    """Extract rank from DiffDock output filename (e.g., rank1_confidence-0.85.sdf)."""
    try:
        if "rank" in filename.lower():
            parts = filename.lower().split("rank")
            if len(parts) > 1:
                rank_str = ""
                for c in parts[1]:
                    if c.isdigit():
                        rank_str += c
                    else:
                        break
                if rank_str:
                    return int(rank_str)
    except:
        pass
    return 0


def prepare_protein_for_diffdock(input_pdb: Path, output_dir: Path) -> Path:
    """
    Prepare a protein PDB file for DiffDock by fixing common issues:
    - Convert non-standard histidine names (HSD, HSE, HSP) to standard HIS
    - Remove HETATM records (water, ions, ligands)
    - Keep only ATOM records
    
    Args:
        input_pdb: Path to input PDB file
        output_dir: Directory to save prepared PDB
    
    Returns:
        Path to prepared PDB file
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    output_pdb = output_dir / f"{input_pdb.stem}_prepared.pdb"
    
    # Residue name mapping (CHARMM/NAMD histidine variants -> standard)
    residue_mapping = {
        'HSD': 'HIS',  # delta-protonated histidine
        'HSE': 'HIS',  # epsilon-protonated histidine  
        'HSP': 'HIS',  # doubly protonated histidine
        'HIE': 'HIS',  # AMBER epsilon-protonated
        'HID': 'HIS',  # AMBER delta-protonated
        'HIP': 'HIS',  # AMBER doubly protonated
    }
    
    with open(input_pdb, 'r') as f_in, open(output_pdb, 'w') as f_out:
        for line in f_in:
            # Only keep ATOM records (skip HETATM, waters, etc.)
            if line.startswith('ATOM'):
                # Fix non-standard residue names
                res_name = line[17:20].strip()
                if res_name in residue_mapping:
                    # Replace residue name in the line (columns 18-20)
                    new_res = residue_mapping[res_name]
                    line = line[:17] + f"{new_res:>3}" + line[20:]
                f_out.write(line)
            elif line.startswith(('END', 'TER')):
                f_out.write(line)
    
    return output_pdb


print("Helper functions defined.")

Helper functions defined.


In [5]:
# ============================================================================
# DIFFDOCK INFERENCE FUNCTIONS
# ============================================================================

def create_protein_ligand_csv(
    proteins: List[Path],
    ligands: List[Path],
    output_csv: Path,
) -> List[Tuple[str, Path, Path]]:
    """
    Create a CSV file for DiffDock batch inference.
    
    CSV format: complex_name, protein_path, ligand_description, protein_sequence
    """
    combinations = []
    
    with open(output_csv, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['complex_name', 'protein_path', 'ligand_description', 'protein_sequence'])
        
        for protein in proteins:
            protein_name = get_file_stem(protein)
            for ligand in ligands:
                ligand_name = get_file_stem(ligand)
                complex_name = f"{ligand_name}__{protein_name}"
                
                writer.writerow([
                    complex_name,
                    str(protein.absolute()),
                    str(ligand.absolute()),
                    ''  # Empty protein_sequence since we have PDB files
                ])
                combinations.append((complex_name, protein, ligand))
    
    print(f"Created CSV with {len(combinations)} combinations: {output_csv}")
    return combinations


def run_diffdock_batch(
    csv_path: Path,
    output_dir: Path,
    samples: int = NUM_SAMPLES,
    device: str = DIFFDOCK_DEVICE,
) -> Tuple[bool, str]:
    """
    Run DiffDock inference using CSV input for batch processing.
    
    Command:
    conda run -n diffdock python -m inference \
        --config default_inference_args.yaml \
        --protein_ligand_csv input.csv \
        --out_dir ./results \
        --samples N
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    
    cmd = [
        "conda", "run", "-n", DIFFDOCK_CONDA_ENV, "--no-capture-output",
        "python", "-m", "inference",
        "--config", "default_inference_args.yaml",
        "--protein_ligand_csv", str(csv_path),
        "--out_dir", str(output_dir),
        "--samples", str(samples),
        "--save_visualisation",  # Save SDF files
    ]
    
    print(f"Running DiffDock batch inference...")
    print(f"  Command: {' '.join(cmd)}")
    print(f"  Working dir: {DIFFDOCK_DIR}")
    
    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=7200,  # 2 hour timeout for batch
            cwd=str(DIFFDOCK_DIR),
        )
        
        if result.returncode == 0:
            return True, ""
        else:
            error_msg = f"Return code {result.returncode}. stderr: {result.stderr[-1000:] if result.stderr else 'None'}"
            return False, error_msg
            
    except subprocess.TimeoutExpired:
        return False, "Batch docking timeout (7200s)"
    except Exception as e:
        return False, str(e)


def run_diffdock_single(
    protein_path: Path,
    ligand_path: Path,
    output_dir: Path,
    samples: int = NUM_SAMPLES,
    device: str = DIFFDOCK_DEVICE,
) -> Tuple[bool, List[Path], str]:
    """
    Run DiffDock inference for a single protein-ligand pair.
    
    Command:
    conda run -n diffdock python -m inference \
        --protein_path protein.pdb \
        --ligand ligand.sdf \
        --out_dir ./output \
        --samples N
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    
    cmd = [
        "conda", "run", "-n", DIFFDOCK_CONDA_ENV, "--no-capture-output",
        "python", "-m", "inference",
        "--config", "default_inference_args.yaml",
        "--protein_path", str(protein_path),
        "--ligand", str(ligand_path),
        "--out_dir", str(output_dir),
        "--samples", str(samples),
        "--save_visualisation",
    ]
    
    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=1800,  # 30 minute timeout per docking
            cwd=str(DIFFDOCK_DIR),
        )
        
        # Find output SDF files
        output_sdfs = []
        for sdf in output_dir.rglob("*.sdf"):
            if sdf.stat().st_size > 0:
                output_sdfs.append(sdf)
        
        output_sdfs = sorted(output_sdfs, key=lambda p: extract_rank_from_filename(p.name))
        
        if output_sdfs:
            return True, output_sdfs, ""
        
        error_msg = f"No output SDF files found. stderr: {result.stderr[-500:] if result.stderr else ''}"
        return False, [], error_msg
        
    except subprocess.TimeoutExpired:
        return False, [], "Docking timeout (1800s)"
    except Exception as e:
        return False, [], str(e)


print("DiffDock inference functions defined.")

DiffDock inference functions defined.


In [6]:
# ============================================================================
# BATCH DOCKING FUNCTIONS
# ============================================================================

def dock_single_combination(
    protein_path: Path,
    ligand_path: Path,
    output_base_dir: Path,
    samples: int = NUM_SAMPLES,
    device: str = DIFFDOCK_DEVICE,
) -> DockingResult:
    """Dock a single protein-ligand combination."""
    protein_name = get_file_stem(protein_path)
    ligand_name = get_file_stem(ligand_path)
    combo_name = f"{ligand_name}__{protein_name}"
    output_dir = output_base_dir / combo_name
    
    result = DockingResult(
        protein_name=protein_name,
        ligand_name=ligand_name,
        protein_path=protein_path,
        ligand_path=ligand_path,
        output_dir=output_dir,
        status="pending",
    )
    
    # Check if already exists
    if not OVERWRITE_EXISTING and output_dir.exists():
        existing_sdfs = list(output_dir.rglob("*.sdf"))
        if existing_sdfs:
            result.status = "skipped"
            result.num_poses = len(existing_sdfs)
            result.pose_files = sorted(existing_sdfs, key=lambda p: extract_rank_from_filename(p.name))
            return result
    
    start_time = time.time()
    
    # Prepare protein (fix non-standard residue names like HSD/HSE/HSP -> HIS)
    prepared_dir = output_base_dir / "prepared_proteins"
    prepared_protein = prepare_protein_for_diffdock(protein_path, prepared_dir)
    print(f"  Prepared protein: {prepared_protein.name}")
    
    success, pose_files, error = run_diffdock_single(
        protein_path=prepared_protein,
        ligand_path=ligand_path,
        output_dir=output_dir,
        samples=samples,
        device=device,
    )
    
    result.elapsed_time = time.time() - start_time
    
    if success and pose_files:
        result.status = "success"
        result.num_poses = len(pose_files)
        result.pose_files = pose_files
    else:
        result.status = "failed"
        result.error_message = error
    
    return result


def run_diffdock_sequential(
    proteins: List[Path],
    ligands: List[Path],
    output_dir: Path,
    samples: int = NUM_SAMPLES,
    device: str = DIFFDOCK_DEVICE,
) -> List[DockingResult]:
    """
    Run DiffDock for all protein-ligand combinations sequentially.
    Each combination is docked one at a time.
    """
    total = len(proteins) * len(ligands)
    results = []
    
    print("=" * 80)
    print("DiffDock Sequential Docking")
    print("=" * 80)
    print(f"Proteins: {len(proteins)}")
    print(f"Ligands: {len(ligands)}")
    print(f"Total combinations: {total}")
    print(f"Samples per complex: {samples}")
    print()
    
    idx = 0
    for protein in proteins:
        for ligand in ligands:
            idx += 1
            print(f"\n[{idx}/{total}] {get_file_stem(ligand)} + {get_file_stem(protein)}")
            
            result = dock_single_combination(
                protein_path=protein,
                ligand_path=ligand,
                output_base_dir=output_dir,
                samples=samples,
                device=device,
            )
            
            results.append(result)
            
            status_icon = {"success": "✓", "failed": "✗", "skipped": "⊘"}.get(result.status, "?")
            print(f"  {status_icon} {result.status} | Poses: {result.num_poses} | Time: {result.elapsed_time:.1f}s")
            
            if result.error_message:
                print(f"    Error: {result.error_message[:100]}")
    
    return results


print("Batch docking functions defined.")

Batch docking functions defined.


In [7]:
# ============================================================================
# SUMMARY FUNCTIONS
# ============================================================================

def generate_summary(results: List[DockingResult]) -> Dict:
    """Generate summary of docking results."""
    summary = {
        "timestamp": datetime.now().isoformat(),
        "configuration": {
            "num_samples": NUM_SAMPLES,
            "device": DIFFDOCK_DEVICE,
            "diffdock_dir": str(DIFFDOCK_DIR),
        },
        "overall": {
            "total_combinations": len(results),
            "successful": sum(1 for r in results if r.status == "success"),
            "failed": sum(1 for r in results if r.status == "failed"),
            "skipped": sum(1 for r in results if r.status == "skipped"),
            "total_poses": sum(r.num_poses for r in results),
            "total_time_seconds": sum(r.elapsed_time for r in results),
        },
        "by_protein": defaultdict(lambda: {"combinations": 0, "poses": 0, "success": 0}),
        "by_ligand": defaultdict(lambda: {"combinations": 0, "poses": 0, "success": 0}),
        "combinations": [],
    }
    
    for r in results:
        summary["by_protein"][r.protein_name]["combinations"] += 1
        summary["by_protein"][r.protein_name]["poses"] += r.num_poses
        if r.status == "success":
            summary["by_protein"][r.protein_name]["success"] += 1
            
        summary["by_ligand"][r.ligand_name]["combinations"] += 1
        summary["by_ligand"][r.ligand_name]["poses"] += r.num_poses
        if r.status == "success":
            summary["by_ligand"][r.ligand_name]["success"] += 1
        
        summary["combinations"].append({
            "protein": r.protein_name,
            "ligand": r.ligand_name,
            "status": r.status,
            "num_poses": r.num_poses,
            "time_seconds": round(r.elapsed_time, 2),
            "error": r.error_message if r.error_message else None,
        })
    
    summary["by_protein"] = dict(summary["by_protein"])
    summary["by_ligand"] = dict(summary["by_ligand"])
    
    return summary


def print_summary(summary: Dict):
    """Print formatted docking summary."""
    print("\n" + "=" * 80)
    print("DIFFDOCK DOCKING SUMMARY")
    print("=" * 80)
    
    overall = summary["overall"]
    print(f"\nOverall Statistics:")
    print(f"  Total combinations: {overall['total_combinations']}")
    print(f"  Successful: {overall['successful']}")
    print(f"  Failed: {overall['failed']}")
    print(f"  Skipped: {overall['skipped']}")
    print(f"  Total poses generated: {overall['total_poses']}")
    print(f"  Total time: {overall['total_time_seconds']:.1f}s ({overall['total_time_seconds']/60:.1f} min)")
    
    print("\n" + "-" * 80)
    print("Results by Protein:")
    print("-" * 80)
    for protein, stats in sorted(summary["by_protein"].items()):
        print(f"  {protein:40s} | Combos: {stats['combinations']:3d} | Poses: {stats['poses']:4d}")
    
    print("\n" + "-" * 80)
    print("Results by Ligand:")
    print("-" * 80)
    for ligand, stats in sorted(summary["by_ligand"].items()):
        print(f"  {ligand:40s} | Combos: {stats['combinations']:3d} | Poses: {stats['poses']:4d}")


print("Summary functions defined.")

Summary functions defined.


In [8]:
# ============================================================================
# COLLECT INPUT FILES
# ============================================================================

# Ligands: SDF, MOL2, PDB files from drugs_dir
# Proteins: PDB files from receptors_dir (only those ending with a number)

ligand_files = collect_files(drugs_dir, [".sdf", ".mol2", ".pdb"])
all_receptor_files = collect_files(receptors_dir, [".pdb"])

# Filter to only keep receptor files whose stem ends with a digit (e.g., Fr300, Fr0)
receptor_files = [f for f in all_receptor_files if f.stem[-1].isdigit()]
print(f"Filtered receptors: {len(receptor_files)} of {len(all_receptor_files)} (keeping only files ending with a number)")

print("=" * 80)
print("INPUT FILES FOR DIFFDOCK")
print("=" * 80)

print(f"\nProteins ({len(receptor_files)} files from {receptors_dir}):")
for pdb in receptor_files:
    print(f"  - {pdb.name}")

print(f"\nLigands ({len(ligand_files)} files from {drugs_dir}):")
for lig in ligand_files:
    print(f"  - {lig.name}")

print(f"\nTotal docking combinations: {len(receptor_files) * len(ligand_files)}")
print(f"Samples per complex: {NUM_SAMPLES}")
print(f"Expected total poses: {len(receptor_files) * len(ligand_files) * NUM_SAMPLES}")

Filtered receptors: 4 of 4 (keeping only files ending with a number)
INPUT FILES FOR DIFFDOCK

Proteins (4 files from /home/manndo/MasterProject/Orai):
  - Orai1WT-MDSnap-Fr300.pdb
  - Orai1WT-MDSnap-Fr400.pdb
  - Orai1WT-MDSnap-Fr499.pdb
  - Orai1WT-START-Fr0.pdb

Ligands (5 files from /home/manndo/MasterProject/Drugs):
  - 2abp-nh2-OPT.pdb
  - 2abp-nh3p-OPT.pdb
  - Synta-66-OPT-Singlet.pdb
  - gsk7975a-deprot-OPT.pdb
  - gsk7975a-prot-OPT.pdb

Total docking combinations: 20
Samples per complex: 30
Expected total poses: 600


In [9]:
# ============================================================================
# RUN DIFFDOCK DOCKING
# ============================================================================

# Run docking sequentially for each protein-ligand combination
diffdock_results = run_diffdock_sequential(
    proteins=receptor_files,
    ligands=ligand_files,
    output_dir=DIFFDOCK_OUTPUT_DIR,
    samples=NUM_SAMPLES,
    device=DIFFDOCK_DEVICE,
)

# Generate and display summary
docking_summary = generate_summary(diffdock_results)
print_summary(docking_summary)

# Save summary to file
summary_path = DIFFDOCK_OUTPUT_DIR / "docking_summary.json"
with open(summary_path, 'w') as f:
    json.dump(docking_summary, f, indent=2)
print(f"\nSummary saved to: {summary_path}")

DiffDock Sequential Docking
Proteins: 4
Ligands: 5
Total combinations: 20
Samples per complex: 30


[1/20] 2abp-nh2-OPT + Orai1WT-MDSnap-Fr300
  Prepared protein: Orai1WT-MDSnap-Fr300_prepared.pdb
  ✓ success | Poses: 11 | Time: 186.4s

[2/20] 2abp-nh3p-OPT + Orai1WT-MDSnap-Fr300
  Prepared protein: Orai1WT-MDSnap-Fr300_prepared.pdb
  ✓ success | Poses: 11 | Time: 146.6s

[3/20] Synta-66-OPT-Singlet + Orai1WT-MDSnap-Fr300
  Prepared protein: Orai1WT-MDSnap-Fr300_prepared.pdb
  ✓ success | Poses: 11 | Time: 247.6s

[4/20] gsk7975a-deprot-OPT + Orai1WT-MDSnap-Fr300
  Prepared protein: Orai1WT-MDSnap-Fr300_prepared.pdb
  ✓ success | Poses: 11 | Time: 278.9s

[5/20] gsk7975a-prot-OPT + Orai1WT-MDSnap-Fr300
  Prepared protein: Orai1WT-MDSnap-Fr300_prepared.pdb
  ✓ success | Poses: 11 | Time: 257.3s

[6/20] 2abp-nh2-OPT + Orai1WT-MDSnap-Fr400
  Prepared protein: Orai1WT-MDSnap-Fr400_prepared.pdb
  ✓ success | Poses: 11 | Time: 183.3s

[7/20] 2abp-nh3p-OPT + Orai1WT-MDSnap-Fr400
  Prepared pr

In [10]:
# ============================================================================
# VISUALIZE RESULTS
# ============================================================================
import pandas as pd

# Convert results to DataFrame
results_data = []
for r in diffdock_results:
    for i, pose_file in enumerate(r.pose_files):
        confidence = extract_confidence_from_filename(pose_file.name)
        rank = extract_rank_from_filename(pose_file.name)
        results_data.append({
            "protein": r.protein_name,
            "ligand": r.ligand_name,
            "rank": rank if rank > 0 else i + 1,
            "confidence": confidence,
            "sdf_path": str(pose_file),
            "status": r.status,
        })

results_df = pd.DataFrame(results_data)

print("=" * 80)
print("DOCKING RESULTS DATAFRAME")
print("=" * 80)
print(f"\nTotal poses: {len(results_df)}")

if not results_df.empty:
    print("\nPoses per protein-ligand combination:")
    pose_counts = results_df.groupby(["protein", "ligand"]).agg({
        "rank": "count",
        "confidence": "mean",
    }).reset_index()
    pose_counts.columns = ["protein", "ligand", "num_poses", "avg_confidence"]
    display(pose_counts)
    
    print("\nConfidence score distribution:")
    if results_df['confidence'].sum() > 0:
        print(f"  Mean confidence: {results_df['confidence'].mean():.3f}")
        print(f"  Max confidence: {results_df['confidence'].max():.3f}")
        print(f"  Min confidence: {results_df['confidence'].min():.3f}")

# Save to CSV
csv_path = DIFFDOCK_OUTPUT_DIR / "diffdock_poses.csv"
results_df.to_csv(csv_path, index=False)
print(f"\nResults saved to: {csv_path}")

DOCKING RESULTS DATAFRAME

Total poses: 209

Poses per protein-ligand combination:


,protein,ligand,num_poses,avg_confidence
0,Orai1WT-MDSnap-Fr300,2abp-nh2-OPT,11,2.724545
1,Orai1WT-MDSnap-Fr300,2abp-nh3p-OPT,11,3.860909
2,Orai1WT-MDSnap-Fr300,Synta-66-OPT-Singlet,11,2.878182
3,Orai1WT-MDSnap-Fr300,gsk7975a-deprot-OPT,11,2.704545
4,Orai1WT-MDSnap-Fr300,gsk7975a-prot-OPT,11,3.227273
5,Orai1WT-MDSnap-Fr400,2abp-nh2-OPT,11,3.701818
6,Orai1WT-MDSnap-Fr400,2abp-nh3p-OPT,11,4.165455
7,Orai1WT-MDSnap-Fr400,Synta-66-OPT-Singlet,11,2.937273
8,Orai1WT-MDSnap-Fr400,gsk7975a-deprot-OPT,11,2.710909
9,Orai1WT-MDSnap-Fr400,gsk7975a-prot-OPT,11,3.233636



Confidence score distribution:
  Mean confidence: 3.096
  Max confidence: 6.200
  Min confidence: 0.000

Results saved to: /home/manndo/MasterProject/diffdock_results/diffdock_poses.csv


In [11]:
# ============================================================================
# LIST ALL GENERATED POSES
# ============================================================================

def list_pose_files(output_dir: Path) -> Dict[str, List[Path]]:
    """List all generated pose files organized by combination."""
    poses_by_combo = {}
    
    if not output_dir.exists():
        return poses_by_combo
    
    for combo_dir in sorted(output_dir.iterdir()):
        if not combo_dir.is_dir():
            continue
        
        # DiffDock creates files like rank1_confidence-0.85.sdf
        pose_files = sorted(combo_dir.rglob("*.sdf"), key=lambda p: extract_rank_from_filename(p.name))
        if pose_files:
            poses_by_combo[combo_dir.name] = pose_files
    
    return poses_by_combo


pose_files = list_pose_files(DIFFDOCK_OUTPUT_DIR)

print("=" * 80)
print("GENERATED POSE FILES")
print("=" * 80)
print(f"\nOutput directory: {DIFFDOCK_OUTPUT_DIR}")
print(f"Total combinations with poses: {len(pose_files)}")
print()

total_poses = 0
for combo_name, files in pose_files.items():
    print(f"\n{combo_name}/")
    for f in files[:5]:  # Show first 5 poses per combination
        conf = extract_confidence_from_filename(f.name)
        print(f"  └── {f.name} (confidence: {conf:.3f})")
    if len(files) > 5:
        print(f"  └── ... and {len(files) - 5} more")
    total_poses += len(files)

print("\n" + "-" * 80)
print(f"TOTAL POSES GENERATED: {total_poses}")
print("-" * 80)

GENERATED POSE FILES

Output directory: /home/manndo/MasterProject/diffdock_results
Total combinations with poses: 19


2abp-nh2-OPT__Orai1WT-MDSnap-Fr300/
  └── rank1.sdf (confidence: 0.000)
  └── rank1_confidence-1.30.sdf (confidence: 1.300)
  └── rank2_confidence-1.31.sdf (confidence: 1.310)
  └── rank3_confidence-1.34.sdf (confidence: 1.340)
  └── rank4_confidence-1.91.sdf (confidence: 1.910)
  └── ... and 6 more

2abp-nh2-OPT__Orai1WT-MDSnap-Fr400/
  └── rank1_confidence-1.60.sdf (confidence: 1.600)
  └── rank1.sdf (confidence: 0.000)
  └── rank2_confidence-2.38.sdf (confidence: 2.380)
  └── rank3_confidence-3.38.sdf (confidence: 3.380)
  └── rank4_confidence-3.42.sdf (confidence: 3.420)
  └── ... and 6 more

2abp-nh2-OPT__Orai1WT-MDSnap-Fr499/
  └── rank1_confidence-1.69.sdf (confidence: 1.690)
  └── rank1.sdf (confidence: 0.000)
  └── rank2_confidence-2.36.sdf (confidence: 2.360)
  └── rank3_confidence-2.76.sdf (confidence: 2.760)
  └── rank4_confidence-3.22.sdf (confidence: 3.2